# Final-B 17-class — Evaluate and register runs

Re-evaluate all 30 finalized checkpoints on validation and test, export prediction/report CSVs, and create one immutable timestamped registry. Each registry row represents one training run; every other report table carries lineage back to those `run_id` values.

In [ ]:
from pathlib import Path
import sys

if Path('/content').exists():
    ROOT = Path('/content/woodbench')
else:
    ROOT = Path.cwd()

sys.path.insert(0, str(ROOT))

In [ ]:
if Path('/content').exists():
    !pip install -q -r {ROOT}/requirements.txt

    DATA_DIR = ROOT / "data" / "task-B-group-split"
    if not DATA_DIR.exists():
        from huggingface_hub import snapshot_download

        snapshot_download(
            repo_id="ny0g507h3p/wood-task-B",
            repo_type="dataset",
            local_dir=str(DATA_DIR),
            ignore_patterns=["*.md", "README*"],
        )

In [ ]:
import gc
import hashlib
import json
import platform
import subprocess
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, cohen_kappa_score,
    confusion_matrix, log_loss, matthews_corrcoef,
    precision_recall_fscore_support, top_k_accuracy_score,
)
from torch.utils.data import DataLoader

from src.data import FlatConiferDataset
from src.data.transforms import get_val_transforms
from src.models import create_model

DATA_DIR = ROOT / "data" / "task-B-group-split"
EXPERIMENT_DIR = ROOT / "data" / "experiments" / "Task-B-new" / "final"
OUTPUT_DIR = ROOT / "data" / "experiments" / "phase_1" / "final_b_new_recomputed"
EVALUATION_NOTEBOOK = "notebooks/phase 1/task-B-17-new/06_evaluate_and_register_runs.ipynb"
REGISTRY_NAME = "task-b-17class"
REGISTRY_CREATED_AT = datetime.now(timezone.utc)
REGISTRY_TIMESTAMP = REGISTRY_CREATED_AT.strftime("%Y%m%dT%H%M%SZ")
RUN_OUTPUT_DIR = OUTPUT_DIR / REGISTRY_TIMESTAMP
REGISTRY_PATH = RUN_OUTPUT_DIR / f"{REGISTRY_NAME}-{REGISTRY_TIMESTAMP}-registry.csv"
PREDICTION_DIR = RUN_OUTPUT_DIR / "predictions"

BACKBONES = ["resnet50", "efficientnet_b3", "mobilenetv3_large", "convnext_tiny", "swin_tiny"]
MAIN_SEEDS = [42, 123, 456]
ABLATIONS = ["taskB_90rotation", "taskB_label_smoothing_0.0", "taskB_label_smoothing_0.2"]
TRAINING_NOTEBOOKS = {
    "resnet50": "notebooks/phase 1/task-B-17-new/01_train_resnet50.ipynb",
    "efficientnet_b3": "notebooks/phase 1/task-B-17-new/02_train_efficientnet_b3.ipynb",
    "mobilenetv3_large": "notebooks/phase 1/task-B-17-new/03_train_mobilenetv3_large.ipynb",
    "convnext_tiny": "notebooks/phase 1/task-B-17-new/04_train_convnext_tiny.ipynb",
    "swin_tiny": "notebooks/phase 1/task-B-17-new/05_train_swin_tiny.ipynb",
}
BACKBONE_BATCH_SIZES = {"resnet50": 64, "efficientnet_b3": 32, "mobilenetv3_large": 64, "convnext_tiny": 32, "swin_tiny": 32}
EXPERIMENT_DEFAULTS = {
    "main": {"label_smoothing": 0.1, "use_90rotation": False},
    "taskB_90rotation": {"label_smoothing": 0.1, "use_90rotation": True},
    "taskB_label_smoothing_0.0": {"label_smoothing": 0.0, "use_90rotation": False},
    "taskB_label_smoothing_0.2": {"label_smoothing": 0.2, "use_90rotation": False},
}
EXCLUDED_CLASSES = {"Calocedrus sp", "Cupressus funebris (Cupressus pendula)", "Pinus sp"}
EXPECTED_SPLIT_COUNTS = {"train": 3365, "val": 903, "test": 532}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}
IMAGE_SIZE = 224
EVAL_BATCH_SIZE = 64
NUM_WORKERS = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(DEVICE)}")
print(f"Checkpoints: {EXPERIMENT_DIR}")
print(f"Run output: {RUN_OUTPUT_DIR}")
print(f"Registry: {REGISTRY_PATH.name}")

In [ ]:
def relative_to_root(path: Path) -> str:
    return Path(path).resolve().relative_to(ROOT.resolve()).as_posix()


def current_git_commit(root: Path) -> str:
    try:
        return subprocess.run(
            ["git", "rev-parse", "HEAD"], cwd=root, check=True,
            capture_output=True, text=True,
        ).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return "unknown"


def validate_dataset(data_dir: Path) -> tuple[list[str], dict[str, int], str]:
    split_classes, split_counts, manifest_rows = {}, {}, []
    for split, expected_count in EXPECTED_SPLIT_COUNTS.items():
        split_dir = data_dir / split
        if not split_dir.exists():
            raise FileNotFoundError(f"Missing Final-B split: {split_dir}")
        classes = sorted(path.name for path in split_dir.iterdir() if path.is_dir())
        image_paths = sorted(
            path for path in split_dir.rglob("*")
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        )
        actual_count = len(image_paths)
        assert len(classes) == 17, f"{split}: expected 17 classes, got {len(classes)}"
        assert EXCLUDED_CLASSES.isdisjoint(classes), f"{split}: excluded class remains"
        assert actual_count == expected_count, f"{split}: expected {expected_count} images, got {actual_count}"
        split_classes[split], split_counts[split] = classes, actual_count
        for path in image_paths:
            with path.open("rb") as image_file:
                image_digest = hashlib.file_digest(image_file, "sha256").hexdigest()
            manifest_rows.append(f"{relative_to_root(path)}\t{image_digest}")

    reference = split_classes["train"]
    for split in ["val", "test"]:
        assert split_classes[split] == reference, f"{split}: class mapping differs from train"
    digest = hashlib.sha256("\n".join(manifest_rows).encode("utf-8")).hexdigest()
    return reference, split_counts, f"{data_dir.name}:{digest[:16]}"


def make_run_id(experiment: str, seed: int, backbone: str) -> str:
    return f"taskB17__{experiment}__seed_{seed}__{backbone}".replace(".", "p")


def discover_checkpoints(experiment_dir: Path) -> pd.DataFrame:
    records = []
    for checkpoint in sorted(experiment_dir.rglob("*_best.pth")):
        parts = checkpoint.relative_to(experiment_dir).parts
        if parts[0] == "main":
            experiment, seed, backbone = "main", int(parts[1].removeprefix("seed_")), parts[2]
        else:
            experiment, seed, backbone = parts[0], 42, parts[1]
        records.append({
            "run_id": make_run_id(experiment, seed, backbone),
            "experiment": experiment, "seed": seed, "backbone": backbone,
            "checkpoint": checkpoint,
        })

    inventory = pd.DataFrame(records)
    expected = {
        *(f"main|{seed}|{backbone}" for seed in MAIN_SEEDS for backbone in BACKBONES),
        *(f"{experiment}|42|{backbone}" for experiment in ABLATIONS for backbone in BACKBONES),
    }
    actual = {f"{row.experiment}|{row.seed}|{row.backbone}" for row in inventory.itertuples(index=False)}
    assert len(inventory) == 30, f"Expected 30 checkpoints, got {len(inventory)}"
    assert inventory["run_id"].is_unique, "run_id must be unique"
    assert actual == expected, f"Checkpoint mismatch. Missing={sorted(expected - actual)}, extra={sorted(actual - expected)}"
    return inventory.sort_values(["experiment", "seed", "backbone"]).reset_index(drop=True)


GIT_COMMIT = current_git_commit(ROOT)
class_names, split_counts, split_version = validate_dataset(DATA_DIR)
checkpoint_inventory = discover_checkpoints(EXPERIMENT_DIR)
print(f"Dataset: {len(class_names)} classes, split={split_version}, counts={split_counts}")
print(f"Git commit: {GIT_COMMIT}")
print(f"Checkpoint inventory: {len(checkpoint_inventory)} runs")
checkpoint_inventory[["run_id", "experiment", "seed", "backbone"]]

In [ ]:
transform = get_val_transforms(img_size=IMAGE_SIZE)
datasets = {
    split: FlatConiferDataset(DATA_DIR, split=split, transform=transform, task="taskB", classes=class_names)
    for split in ["val", "test"]
}
loaders = {
    split: DataLoader(
        dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda", persistent_workers=NUM_WORKERS > 0,
    )
    for split, dataset in datasets.items()
}


def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}


def load_checkpoint(path: Path):
    try:
        payload = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        payload = torch.load(path, map_location="cpu")
    return payload, payload.get("model_state_dict", payload)


def checkpoint_metadata(payload: dict) -> dict:
    metrics = payload.get("metrics", {})
    best_epoch = payload.get("best_epoch", payload.get("epoch", -1))
    return {
        "checkpoint_best_val_acc": float(payload.get("best_val_acc", metrics.get("val_acc", np.nan))),
        "checkpoint_best_epoch": int(best_epoch) + 1,
        "checkpoint_saved_epoch": int(payload.get("epoch", -1)) + 1,
    }


def run_metadata(record, payload: dict) -> dict:
    run_config_path = record.checkpoint.parent / "run_config.json"
    training_metrics_path = record.checkpoint.parent / "test_metrics.json"
    run_config, training_metrics = load_json(run_config_path), load_json(training_metrics_path)
    defaults = EXPERIMENT_DEFAULTS[record.experiment]
    use_90rotation = bool(run_config.get("use_90rotation", defaults["use_90rotation"]))
    augmentation = (
        "resize224|hflip0.5|vflip0.5|random_90|color_jitter0.2" if use_90rotation
        else "resize224|hflip0.5|vflip0.5|random_rotation30|color_jitter0.2"
    )
    checkpoint_time = datetime.fromtimestamp(record.checkpoint.stat().st_mtime, tz=timezone.utc).isoformat()
    configured_run_date = run_config.get("run_date")
    prediction_path = PREDICTION_DIR / f"{record.run_id}-predictions.csv"
    return {
        "run_id": record.run_id,
        "run_date": configured_run_date or checkpoint_time,
        "run_date_source": "run_config" if configured_run_date else "checkpoint_mtime_utc",
        "notebook": run_config.get("notebook", TRAINING_NOTEBOOKS[record.backbone]),
        "git_commit": run_config.get("git_commit", GIT_COMMIT),
        "git_commit_source": "run_config" if run_config.get("git_commit") else "evaluation_checkout",
        "split_version": split_version,
        "num_classes": len(class_names),
        "n_train": split_counts["train"], "n_val": split_counts["val"], "n_test": split_counts["test"],
        "seed": int(record.seed),
        "lr": float(run_config.get("lr", run_config.get("lr_head", 1e-4))),
        "lr_backbone": float(run_config.get("lr_backbone", 1e-5)),
        "epochs": int(run_config.get("epochs", 50)),
        "epochs_trained": int(training_metrics.get("epochs_trained", int(payload.get("epoch", -1)) + 1)),
        "batch": int(run_config.get("batch", run_config.get("batch_size", BACKBONE_BATCH_SIZES[record.backbone]))),
        "image_size": int(run_config.get("image_size", run_config.get("img_size", IMAGE_SIZE))),
        "label_smoothing": float(run_config.get("label_smoothing", defaults["label_smoothing"])),
        "augmentation": run_config.get("augmentation", augmentation),
        "experiment": record.experiment, "backbone": record.backbone,
        "checkpoint_path": relative_to_root(record.checkpoint),
        "prediction_path": relative_to_root(prediction_path),
        "run_config_path": relative_to_root(run_config_path) if run_config_path.exists() else "",
        "evaluation_notebook": EVALUATION_NOTEBOOK,
        **checkpoint_metadata(payload),
    }


preflight_record = checkpoint_inventory.iloc[0]
preflight_payload, preflight_state = load_checkpoint(preflight_record.checkpoint)
preflight_model = create_model(preflight_record.backbone, num_classes=len(class_names), pretrained=False).to(DEVICE)
preflight_model.load_state_dict(preflight_state, strict=True)
preflight_model.eval()
preflight_images, _ = next(iter(loaders["val"]))
with torch.inference_mode():
    preflight_logits = preflight_model(preflight_images.to(DEVICE))
assert preflight_logits.shape == (len(preflight_images), len(class_names))
print(f"Preflight passed: logits={tuple(preflight_logits.shape)}")
del preflight_model, preflight_payload, preflight_state, preflight_images, preflight_logits
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

In [ ]:
METRIC_COLUMNS = [
    "log_loss", "accuracy", "balanced_accuracy",
    "precision_macro", "recall_macro", "f1_macro",
    "precision_micro", "recall_micro", "f1_micro",
    "precision_weighted", "recall_weighted", "f1_weighted",
    "matthews_corrcoef", "cohen_kappa", "top3_accuracy",
]


@torch.inference_mode()
def infer(model: torch.nn.Module, loader: DataLoader):
    labels, predictions, probabilities = [], [], []
    for images, targets in loader:
        logits = model(images.to(DEVICE, non_blocking=True))
        probs = torch.softmax(logits, dim=1)
        labels.append(targets.numpy())
        predictions.append(probs.argmax(dim=1).cpu().numpy())
        probabilities.append(probs.cpu().numpy())
    return np.concatenate(labels), np.concatenate(predictions), np.concatenate(probabilities)


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_prob: np.ndarray):
    label_ids = np.arange(len(class_names))
    aggregate = {
        "log_loss": float(log_loss(y_true, y_prob, labels=label_ids)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "matthews_corrcoef": float(matthews_corrcoef(y_true, y_pred)),
        "cohen_kappa": float(cohen_kappa_score(y_true, y_pred)),
        "top3_accuracy": float(top_k_accuracy_score(y_true, y_prob, k=3, labels=label_ids)),
    }
    for average in ["macro", "micro", "weighted"]:
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, labels=label_ids, average=average, zero_division=0,
        )
        aggregate[f"precision_{average}"] = float(precision)
        aggregate[f"recall_{average}"] = float(recall)
        aggregate[f"f1_{average}"] = float(f1)

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=label_ids, average=None, zero_division=0
    )
    per_class = pd.DataFrame({
        "class_index": label_ids, "class_name": class_names,
        "precision": precision, "recall": recall, "f1": f1,
        "support": support.astype(int),
    })
    return aggregate, per_class, confusion_matrix(y_true, y_pred, labels=label_ids)


def prediction_table(dataset, y_true, y_pred, y_prob) -> pd.DataFrame:
    image_paths = [relative_to_root(path) for path, _ in dataset.samples]
    top3_indices = np.argsort(y_prob, axis=1)[:, -3:][:, ::-1]
    return pd.DataFrame({
        "image_path": image_paths, "true_index": y_true,
        "true_class": [class_names[index] for index in y_true],
        "predicted_index": y_pred,
        "predicted_class": [class_names[index] for index in y_pred],
        "confidence": y_prob.max(axis=1), "correct": y_true == y_pred,
        "top3_classes": [" | ".join(class_names[index] for index in indices) for indices in top3_indices],
    })

In [ ]:
split_metric_rows, registry_rows = [], []
per_class_frames, prediction_frames, confusion_matrices = [], [], {}

for record in checkpoint_inventory.itertuples(index=False):
    payload, state_dict = load_checkpoint(record.checkpoint)
    model = create_model(record.backbone, num_classes=len(class_names), pretrained=False)
    model.load_state_dict(state_dict, strict=True)
    model = model.to(DEVICE).eval()
    registry_row = run_metadata(record, payload)

    for split, loader in loaders.items():
        y_true, y_pred, y_prob = infer(model, loader)
        aggregate, per_class, matrix = compute_metrics(y_true, y_pred, y_prob)
        confusion_key = f"{record.run_id}__{split}"
        common = {
            "run_id": record.run_id, "experiment": record.experiment,
            "seed": int(record.seed), "backbone": record.backbone, "split": split,
            "num_classes": len(class_names), "num_images": len(y_true),
            "checkpoint_path": registry_row["checkpoint_path"],
            "prediction_path": registry_row["prediction_path"],
            "confusion_matrix_key": confusion_key,
        }
        split_metric_rows.append({**common, **aggregate})
        per_class_frames.append(per_class.assign(**common))
        prediction_frames.append(prediction_table(datasets[split], y_true, y_pred, y_prob).assign(**common))
        confusion_matrices[confusion_key] = matrix
        registry_row.update({f"{split}_{metric}": aggregate[metric] for metric in METRIC_COLUMNS})
        print(
            f"{record.experiment:<27s} | seed={record.seed:<3d} | {record.backbone:<20s} | "
            f"{split:<4s} | Acc={aggregate['accuracy'] * 100:6.2f}% | F1={aggregate['f1_macro'] * 100:6.2f}%"
        )

    registry_rows.append(registry_row)
    del model, payload, state_dict
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

run_metrics = pd.DataFrame(split_metric_rows)
registry = pd.DataFrame(registry_rows)
per_class_metrics = pd.concat(per_class_frames, ignore_index=True)
predictions = pd.concat(prediction_frames, ignore_index=True)
REQUIRED_REGISTRY_COLUMNS = [
    "run_id", "run_date", "notebook", "git_commit", "split_version",
    "num_classes", "n_train", "n_val", "n_test", "seed", "lr",
    "epochs", "batch", "image_size", "label_smoothing", "augmentation",
    "checkpoint_path", "prediction_path",
    *(f"{split}_{metric}" for split in ["val", "test"] for metric in METRIC_COLUMNS),
]
missing_columns = sorted(set(REQUIRED_REGISTRY_COLUMNS) - set(registry.columns))
assert not missing_columns, f"Registry missing columns: {missing_columns}"
assert len(registry) == 30 and registry["run_id"].is_unique
assert len(run_metrics) == 60
registry = registry[REQUIRED_REGISTRY_COLUMNS + [c for c in registry.columns if c not in REQUIRED_REGISTRY_COLUMNS]]
registry

In [ ]:
main_test = run_metrics.query("experiment == 'main' and split == 'test'")
summary_rows = []
for backbone, group in main_test.groupby("backbone", sort=False):
    summary = {
        "backbone": backbone, "runs": len(group),
        "source_run_ids": "|".join(sorted(group["run_id"])),
    }
    for metric in METRIC_COLUMNS:
        summary[f"{metric}_mean"] = group[metric].mean()
        summary[f"{metric}_std"] = group[metric].std(ddof=0)
    summary_rows.append(summary)
main_test_summary = pd.DataFrame(summary_rows)

baseline = main_test.query("seed == 42")[["run_id", "backbone", *METRIC_COLUMNS]].rename(
    columns={"run_id": "baseline_run_id", **{m: f"main_seed42_{m}" for m in METRIC_COLUMNS}}
)
ablation_comparison = run_metrics.query("experiment != 'main' and split == 'test'").merge(
    baseline, on="backbone", validate="many_to_one"
)
for metric in METRIC_COLUMNS:
    ablation_comparison[f"{metric}_delta_vs_main"] = (
        ablation_comparison[metric] - ablation_comparison[f"main_seed42_{metric}"]
    )

inventory_to_save = checkpoint_inventory.assign(
    checkpoint_path=checkpoint_inventory["checkpoint"].map(relative_to_root)
).drop(columns="checkpoint")
valid_run_ids = set(registry["run_id"])
for table_name, table in {
    "checkpoint_inventory": inventory_to_save, "split_metrics": run_metrics,
    "per_class_metrics": per_class_metrics, "predictions": predictions,
}.items():
    assert "run_id" in table and set(table["run_id"]) <= valid_run_ids, f"Bad lineage: {table_name}"
for source_ids in main_test_summary["source_run_ids"]:
    assert set(source_ids.split("|")) <= valid_run_ids
assert set(ablation_comparison["run_id"]) <= valid_run_ids
assert set(ablation_comparison["baseline_run_id"]) <= valid_run_ids

lineage_rows = [
    {"report_file": "split_metrics.csv", "report_row": f"{row.run_id}|{row.split}", "run_id": row.run_id, "role": "source"}
    for row in run_metrics.itertuples(index=False)
]
for row in main_test_summary.itertuples(index=False):
    lineage_rows.extend(
        {"report_file": "main_test_summary.csv", "report_row": row.backbone, "run_id": source_id, "role": "source"}
        for source_id in row.source_run_ids.split("|")
    )
for row in ablation_comparison.itertuples(index=False):
    lineage_rows.extend([
        {"report_file": "ablation_vs_main_seed42.csv", "report_row": row.run_id, "run_id": row.run_id, "role": "candidate"},
        {"report_file": "ablation_vs_main_seed42.csv", "report_row": row.run_id, "run_id": row.baseline_run_id, "role": "baseline"},
    ])
report_lineage = pd.DataFrame(lineage_rows)
assert set(report_lineage["run_id"]) <= valid_run_ids

metadata = {
    "registry_created_at_utc": REGISTRY_CREATED_AT.isoformat(),
    "registry_file": relative_to_root(REGISTRY_PATH),
    "evaluation_notebook": EVALUATION_NOTEBOOK, "git_commit": GIT_COMMIT,
    "split_version": split_version,
    "split_version_method": "sha256(relative_image_path + image_sha256)[:16]",
    "device_type": DEVICE.type,
    "device_name": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else platform.processor() or platform.machine(),
    "python_version": platform.python_version(), "torch_version": torch.__version__,
    "classes": class_names, "excluded_classes": sorted(EXCLUDED_CLASSES),
    "split_counts": split_counts, "registered_runs": len(registry),
    "evaluated_splits": ["val", "test"], "metric_columns": METRIC_COLUMNS,
    "lineage_rule": "run_id/source_run_ids/baseline_run_id values resolve to registry rows",
}

if RUN_OUTPUT_DIR.exists():
    raise FileExistsError(f"Refusing to overwrite an existing registry run: {RUN_OUTPUT_DIR}")
PREDICTION_DIR.mkdir(parents=True, exist_ok=False)
registry.to_csv(REGISTRY_PATH, index=False)
inventory_to_save.to_csv(RUN_OUTPUT_DIR / "checkpoint_inventory.csv", index=False)
run_metrics.to_csv(RUN_OUTPUT_DIR / "split_metrics.csv", index=False)
run_metrics.to_json(RUN_OUTPUT_DIR / "split_metrics.json", orient="records", indent=2, force_ascii=False)
per_class_metrics.to_csv(RUN_OUTPUT_DIR / "per_class_metrics.csv", index=False)
predictions.to_csv(RUN_OUTPUT_DIR / "predictions.csv", index=False)
registry_by_id = registry.set_index("run_id")
for run_id, frame in predictions.groupby("run_id", sort=False):
    frame.to_csv(ROOT / registry_by_id.at[run_id, "prediction_path"], index=False)
main_test_summary.to_csv(RUN_OUTPUT_DIR / "main_test_summary.csv", index=False)
ablation_comparison.to_csv(RUN_OUTPUT_DIR / "ablation_vs_main_seed42.csv", index=False)
report_lineage.to_csv(RUN_OUTPUT_DIR / "report_lineage.csv", index=False)
np.savez_compressed(RUN_OUTPUT_DIR / "confusion_matrices.npz", **confusion_matrices)
(RUN_OUTPUT_DIR / "metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8"
)
assert all((ROOT / path).exists() for path in registry["checkpoint_path"])
assert all((ROOT / path).exists() for path in registry["prediction_path"])
print(f"Saved registry and reports to: {RUN_OUTPUT_DIR}")
print(f"Registry rows: {len(registry)} -> {REGISTRY_PATH.name}")
main_test_summary[[
    "backbone", "source_run_ids", "accuracy_mean", "accuracy_std",
    "f1_macro_mean", "f1_macro_std",
]].assign(
    accuracy_mean=lambda x: x.accuracy_mean * 100, accuracy_std=lambda x: x.accuracy_std * 100,
    f1_macro_mean=lambda x: x.f1_macro_mean * 100, f1_macro_std=lambda x: x.f1_macro_std * 100,
).round(3)